# Phase 6 — RAG Sémantique

**Objectif** : remplacer le RAG lexical (TF-IDF + cosinus) par un RAG sémantique basé sur des
embeddings denses multilingues et un index FAISS.

| Composant | Choix | Justification |
|---|---|---|
| Embeddings | `paraphrase-multilingual-mpnet-base-v2` | Multilingual FR+EN, 768d, state-of-the-art similarité de phrases |
| Vector DB | FAISS IndexFlatIP | Local, rapide, sans serveur ; IP sur vecteurs L2-normalisés = cosinus |
| Base de connaissance | 476 tickets résolus | Même base que Phase 4 (lexical) pour comparaison directe |

> **ADR-004 décidé** : FAISS local (pas ChromaDB/Weaviate) — priorité à la simplicité et
> à la reproductibilité locale sans dépendance externe.

## 1. Imports & configuration

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..').resolve()))

with open('../params.yaml', 'r') as f:
    params = yaml.safe_load(f)

ROOT    = Path('..')
FIG_DIR = ROOT / 'notebooks' / 'figures'
FIG_DIR.mkdir(exist_ok=True)

print('Imports OK')
print(f'RAG base  : {params["data"]["rag_path"]}')
print(f'top_k     : {params["rag"]["top_k"]}')

## 2. Base de connaissance (tickets résolus)

In [ ]:
rag_df = pd.read_csv(ROOT / params['data']['rag_path'], encoding='utf-8-sig')

# Harmoniser le nom Categorie si l'encodage est corrompu
if 'Catégorie' not in rag_df.columns:
    for col in rag_df.columns:
        if 'gorie' in col.lower():
            rag_df = rag_df.rename(columns={col: 'Catégorie'})
            break

print(f'Base RAG : {len(rag_df)} tickets')
print()
print('Distribution par catégorie :')
print(rag_df['Catégorie'].value_counts().to_string())

## 3. Construction de l'index sémantique (FAISS)

In [ ]:
from src.rag.semantic_engine import SemanticRAGEngine

MODEL_NAME = 'paraphrase-multilingual-mpnet-base-v2'
TOP_K      = params['rag']['top_k']
THRESHOLD  = params['rag']['similarity_threshold']

sem_engine = SemanticRAGEngine(
    model_name           = MODEL_NAME,
    top_k                = TOP_K,
    similarity_threshold = THRESHOLD,
)
sem_engine.fit(rag_df)

## 4. Tests qualitatifs

Mêmes 4 cas que le notebook RAG lexical (`04_rag.ipynb`) pour comparer directement.

In [ ]:
from src.data.preprocess import clean_text

cas_tests = [
    {'titre': 'Erreur activation Office 365',
     'desc':  'Microsoft Office affiche message produit non activé',
     'categorie': 'Microsoft'},
    {'titre': 'Problème accès Sage X3',
     'desc':  'Impossible de se connecter à l ERP Sage X3 depuis ce matin',
     'categorie': 'Applicatifs'},
    {'titre': 'Sauvegarde échouée',
     'desc':  'Le système de sauvegarde indique espace disque insuffisant',
     'categorie': 'Backup'},
    {'titre': 'Accès internet lent',
     'desc':  'Connexion internet très lente depuis le bureau au 3ème étage',
     'categorie': 'Réseau'},
]

print('TESTS SEMANTIQUES')
print('=' * 70)
for cas in cas_tests:
    query = clean_text(cas['desc'])
    recs  = sem_engine.recommend(query, category_filter=cas['categorie'])
    print(f"\n{'_'*70}")
    print(f"Ticket    : {cas['titre']}")
    print(f"Catégorie  : {cas['categorie']}")
    print(f"Query     : {query[:70]}...")
    print(f"Résultats ({len(recs)}) :")
    if not recs:
        print('  Aucune solution similaire trouvée')
    else:
        for i, r in enumerate(recs, 1):
            fb = ' [FALLBACK]' if r.get('fallback') else ''
            print(f"  [{i}] Score {r['score']:.4f}{fb}")
            print(f"       Resolution : {r['resolution'][:100]}")

## 5. Évaluation quantitative

- **Couverture** : % de tickets avec au moins 1 recommandation au-dessus du seuil
- **Score moyen** : cosinus moyen des top-1 (hors fallback)
- **Cat. OK** : % où la catégorie retrouvée == catégorie attendue

In [ ]:
# Evaluation realiste : query = titre+description SEULEMENT
# (pas text_rag complet qui contient deja la resolution -> score=1.0 artificiel)
# On simule un nouveau ticket entrant : titre + description, sans resolution.
from src.data.preprocess import clean_text as _clean
import numpy as np

resultats_sem = {}

for cat in rag_df['Catégorie'].unique():
    subset = rag_df[rag_df['Catégorie'] == cat].sample(
        min(20, len(rag_df[rag_df['Catégorie'] == cat])), random_state=42
    )
    avec_reco, cat_correct, scores_top1 = 0, 0, []
    for _, row in subset.iterrows():
        # Query = titre + description uniquement
        query = _clean(str(row.get('Titre', '')) + ' ' + str(row.get('Description', '')))
        if not query.strip():
            query = str(row['text_rag'])[:100]
        recs = sem_engine.recommend(query, category_filter=cat)
        if recs:
            avec_reco += 1
            if not recs[0].get('fallback'):
                scores_top1.append(recs[0]['score'])
            if recs[0].get('categorie') == cat:
                cat_correct += 1
    resultats_sem[cat] = {
        'tickets_testes': len(subset),
        'avec_reco':      avec_reco,
        'couverture_pct': round(avec_reco / len(subset) * 100, 1),
        'score_moyen':    round(float(np.mean(scores_top1)), 4) if scores_top1 else 0.0,
        'cat_precision':  round(cat_correct / len(subset) * 100, 1),
    }

print(f"{'Catégorie':20s} | {'Testes':>6} | {'Couvert':>8} | {'Score moy':>9} | {'Cat. OK':>8}")
print('-' * 60)
for cat, res in resultats_sem.items():
    print(f"{cat:20s} | {res['tickets_testes']:>6} | "
          f"{res['couverture_pct']:>7.1f}% | {res['score_moyen']:>9.4f} | "
          f"{res['cat_precision']:>7.1f}%")

global_cov   = np.mean([r['couverture_pct'] for r in resultats_sem.values()])
valid_sc     = [r['score_moyen'] for r in resultats_sem.values() if r['score_moyen'] > 0]
global_score = float(np.mean(valid_sc)) if valid_sc else 0.0
print(f'\nCouverture globale : {global_cov:.1f}%')
print(f'Score moyen top-1  : {global_score:.4f}')
print('(query = titre+description seul, index = text_rag complet)')


## 6. Comparaison Lexical vs Sémantique

In [ ]:
from src.rag.engine import RAGEngine as LexRAGEngine
from src.data.preprocess import clean_text as _clean2
import numpy as np

lex_engine = LexRAGEngine(top_k=TOP_K, similarity_threshold=THRESHOLD)
lex_engine.fit(rag_df)

resultats_lex = {}
for cat in rag_df['Catégorie'].unique():
    subset = rag_df[rag_df['Catégorie'] == cat].sample(
        min(20, len(rag_df[rag_df['Catégorie'] == cat])), random_state=42
    )
    avec_reco, scores_top1 = 0, []
    for _, row in subset.iterrows():
        query = _clean2(str(row.get('Titre', '')) + ' ' + str(row.get('Description', '')))
        if not query.strip():
            query = str(row['text_rag'])[:100]
        recs = lex_engine.recommend(query, category_filter=cat)
        if recs:
            avec_reco += 1
            if not recs[0].get('fallback'):
                scores_top1.append(recs[0]['score'])
    resultats_lex[cat] = {
        'couverture_pct': round(avec_reco / len(subset) * 100, 1),
        'score_moyen':    round(float(np.mean(scores_top1)), 4) if scores_top1 else 0.0,
    }

cats = list(resultats_sem.keys())
sem_scores = [resultats_sem[c]['score_moyen'] for c in cats]
lex_scores = [resultats_lex[c]['score_moyen'] for c in cats]

print(f"{'Catégorie':20s} | {'Lex score':>10} | {'Sem score':>10} | {'Gain':>8}")
print('-' * 58)
for i, cat in enumerate(cats):
    gain = sem_scores[i] - lex_scores[i]
    arrow = '+' if gain > 0.005 else ('-' if gain < -0.005 else '=')
    print(f"{cat:20s} | {lex_scores[i]:>10.4f} | {sem_scores[i]:>10.4f} | {arrow}{abs(gain):>7.4f}")
print(f"\nMoyenne  | {np.mean(lex_scores):>10.4f} | {np.mean(sem_scores):>10.4f} | "
      f"{np.mean(sem_scores)-np.mean(lex_scores):>+8.4f}")
print('(query = titre+description seul — evaluation realiste)')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(cats))
w = 0.35

# Score moyen
ax = axes[0]
ax.bar(x - w/2, lex_scores, w, label='Lexical (TF-IDF)',    color='#5B9BD5', alpha=0.85)
ax.bar(x + w/2, sem_scores, w, label='Semantique (FAISS)', color='#ED7D31', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(cats, rotation=40, ha='right', fontsize=9)
ax.set_ylabel('Score cosinus moyen (top-1)')
ax.set_title('Score moyen top-1 par categorie')
ax.legend(); ax.set_ylim(0, 1.0); ax.grid(axis='y', alpha=0.3)

# Couverture
ax2 = axes[1]
lex_cov_v = [resultats_lex[c]['couverture_pct'] for c in cats]
sem_cov_v = [resultats_sem[c]['couverture_pct'] for c in cats]
ax2.bar(x - w/2, lex_cov_v, w, label='Lexical',    color='#5B9BD5', alpha=0.85)
ax2.bar(x + w/2, sem_cov_v, w, label='Semantique', color='#ED7D31', alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(cats, rotation=40, ha='right', fontsize=9)
ax2.set_ylabel('Couverture (%)')
ax2.set_title('Couverture par categorie')
ax2.legend(); ax2.set_ylim(0, 115); ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Phase 6 : RAG Lexical vs Semantique', fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(FIG_DIR / 'rag6_01_comparaison.png', bbox_inches='tight')
plt.show()
print('Figure sauvegardee : notebooks/figures/rag6_01_comparaison.png')

## 7. Sauvegarde du moteur sémantique

In [ ]:
SAVE_DIR = str(ROOT / 'models' / 'semantic_rag')
sem_engine.save(SAVE_DIR)

# Test rechargement
engine_loaded = SemanticRAGEngine.load(SAVE_DIR)
test_recs = engine_loaded.recommend(
    clean_text('microsoft office erreur activation licence'),
    category_filter='Microsoft'
)
print(f'\nTest rechargement : {len(test_recs)} recommandation(s)')
for r in test_recs:
    print(f"  Score {r['score']:.4f} | {r['resolution'][:80]}")

## 8. Résumé Phase 6

In [ ]:
import json as _json

metadata = {
    'phase':                 'Phase 6 - RAG semantique',
    'model_embeddings':      MODEL_NAME,
    'vector_db':             'FAISS IndexFlatIP (cosinus via L2-norm)',
    'embed_dim':             768,
    'n_tickets_indexed':     sem_engine._index.ntotal,
    'top_k':                 TOP_K,
    'similarity_threshold':  THRESHOLD,
    'couverture_globale_pct': round(float(global_cov), 1),
    'score_moyen_top1_sem':   round(float(global_score), 4),
    'score_moyen_top1_lex':   round(float(np.mean(lex_scores)), 4),
    'gain_score_moyen':       round(float(global_score - np.mean(lex_scores)), 4),
    'index_path':             'models/semantic_rag/',
    'adr004_decision':        'FAISS local - simplicite et reproductibilite sans serveur',
    'prochaine_etape':        'Phase 7 - Evaluation RAGAS + generation avec citations',
}

out_path = ROOT / 'models' / 'semantic_rag_metadata.json'
out_path.write_text(_json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

print('=' * 60)
print('RESUME - PHASE 6 : RAG Semantique')
print('=' * 60)
for k, v in metadata.items():
    print(f'  {k:35s}: {v}')
print(f'\nMetadonnees : {out_path}')